In [15]:
#basic imports and ipython setup
import matplotlib.pyplot as plt
import numpy as np

import os.path

from tqdm import tqdm

import pandas as pd

import matplotlib as mpl

import Functions_14CO as F

In [2]:
def split_Haino(h):
    if '×' in h:
        x,e = h.split('×')
        e = int(e[2:].replace('−','-'))
    else:
        x,e = h,0
    a = np.array(x.split('±'), dtype=float)
    a *= 10**e
    return a

In [3]:
Haino2004 = pd.read_csv('Raw Muon Tables/Haino2004_Table3_raw.csv')
p_range_H04 = np.array([h.split('–') for h in Haino2004['Momentum Range']], dtype=float) # GeV/c
P_avg_mu_pos_H04 = np.array(Haino2004['mu+ P_avg']) # GeV/c
P_avg_mu_neg_H04 = np.array(Haino2004['mu- P_avg']) # GeV/c
F_mu_pos_H04 = np.array([split_Haino(h) for h in Haino2004['mu+ Flux, dF_stat, dF_sys']]) # (m^2 sr s GeV/c)^-1
F_mu_neg_H04 = np.array([split_Haino(h) for h in Haino2004['mu- Flux, dF_stat, dF_sys']]) # (m^2 sr s GeV/c)^-1

In [27]:
df = pd.DataFrame({'p_low':p_range_H04[:,0],
                   'p_high':p_range_H04[:,1],
                   'P_avg_mu+':P_avg_mu_pos_H04,
                   'P_avg_mu-':P_avg_mu_neg_H04,
                   'F_mu+':F_mu_pos_H04[:,0],
                   'dF_stat_mu+':F_mu_pos_H04[:,1],
                   'dF_sys_mu+':F_mu_pos_H04[:,2],
                   'F_mu-':F_mu_neg_H04[:,0],
                   'dF_stat_mu-':F_mu_neg_H04[:,1],
                   'dF_sys_mu-':F_mu_neg_H04[:,2],
})
df.to_csv('Muon Spectrum Tables/Haino2004_Table3.csv', index=False)

In [4]:
DePascale93 = pd.read_csv('Raw Muon Tables/DePascale1993_Table3_raw.csv')
p_min_DP93, p_max_DP93, p_avg_DP93, N_mu_neg_DP93, N_mu_pos_DP93 = np.array([d.split() for d in DePascale93['Rigidity (GV/c) Minimum | Maximum | Median | N Observed Mu- | N Observed Mu+']], dtype=float).T
F_mu_neg_DP93, rel_err_neg_DP93 = np.array([d.replace('(','').replace(')','').split() for d in DePascale93['Differential Flux mu- (m^2 s sr GV/c)^-1 | % Error']], dtype=float).T
rel_err_neg_DP93 = rel_err_neg_DP93/100
F_neg_err_DP93 = F_mu_neg_DP93 * rel_err_neg_DP93
F_mu_pos_DP93, rel_err_pos_DP93 = np.array([d.replace('(','').replace(')','').split() for d in DePascale93['Differential Flux mu+ (m^2 s sr GV/c)^-1 | % Error']], dtype=float).T
rel_err_pos_DP93 = rel_err_pos_DP93/100
F_pos_err_DP93 = F_mu_pos_DP93 * rel_err_pos_DP93

In [28]:
df = pd.DataFrame({'p_low':p_min_DP93,
                   'p_high':p_max_DP93,
                   'p_avg':p_avg_DP93,
                   'N_mu+':N_mu_pos_DP93,
                   'F_mu+':F_mu_pos_DP93,
                   'dF_mu+':F_pos_err_DP93,
                   'N_mu-':N_mu_neg_DP93,
                   'F_mu-':F_mu_neg_DP93,
                   'dF_mu-':F_neg_err_DP93,
})
df.to_csv('Muon Spectrum Tables/DePascale1993_Table3.csv', index=False)

In [5]:
def split_Kremer(k):
    K = k.replace('–',' ').split()
    p_min = float(K[0])
    p_max = float(K[1])
    p_avg = float(K[2])
    if len(K)<23:
        F_pos_94 = float(K[3])
        F_pos_94_err = float(K[5])
        F_neg_94 = float(K[6])
        F_neg_94_err = float(K[8])
        if len(K)==12:
            F_pos_97 = 0.
            F_pos_97_err = 0.
            F_neg_97 = float(K[9])
            F_neg_97_err = float(K[11])
        else:
            F_pos_97 = float(K[9])
            F_pos_97_err = float(K[11])
            F_neg_97 = float(K[12])
            F_neg_97_err = float(K[14])
    else:
        exp_pos_94 = 10.**(int(K[7][-1]) * (-1 if len(K[7])>3 else 1))
        F_pos_94 = float(K[3])*exp_pos_94
        F_pos_94_err = float(K[5])*exp_pos_94
        exp_neg_94 = 10.**(int(K[12][-1]) * (-1 if len(K[12])>3 else 1))
        F_neg_94 = float(K[8])*exp_neg_94
        F_neg_94_err = float(K[10])*exp_neg_94
        exp_pos_97 = 10.**(int(K[17][-1]) * (-1 if len(K[17])>3 else 1))
        F_pos_97 = float(K[13])*exp_pos_97
        F_pos_97_err = float(K[15])*exp_pos_97
        exp_neg_97 = 10.**(int(K[22][-1]) * (-1 if len(K[22])>3 else 1))
        F_neg_97 = float(K[18])*exp_neg_97
        F_neg_97_err = float(K[20])*exp_neg_97
    return [p_min, p_max, p_avg, F_pos_94, F_pos_94_err, F_neg_94, F_neg_94_err, F_pos_97, F_pos_97_err, F_neg_97, F_neg_97_err]

In [6]:
Kremer99 = pd.read_csv('Raw Muon Tables/Kremer1999_Table1_raw.csv')
p_min_K99, p_max_K99, p_avg_K99, F_mu_pos_K99_94, F_pos_err_K99_94, F_mu_neg_K99_94, F_neg_err_K99_94, F_mu_pos_K99_97, F_pos_err_K99_97, F_mu_neg_K99_97, F_neg_err_K99_97 = np.array([split_Kremer(k) for k in Kremer99['Momentum (GeV/c) Min | Max | Mean | CAPRICE94 mu+ (GeV/c m^2 s sr)^-1 | mu- | CAPRICE97 mu+ (GeV/c m^2 s sr)^-1 | mu-']]).T
K99_clip = F_mu_pos_K99_97>0.
F_mu_tot_K99_94 = F_mu_pos_K99_94+F_mu_neg_K99_94
F_mu_err_K99_94 = np.sqrt(F_pos_err_K99_94**2 + F_neg_err_K99_94**2)
p_avg_K99_97 = p_avg_K99[K99_clip]
F_mu_tot_K99_97 = (F_mu_pos_K99_97+F_mu_neg_K99_97)[K99_clip]
F_mu_err_K99_97 = np.sqrt(F_pos_err_K99_97**2 + F_neg_err_K99_97**2)[K99_clip]

In [29]:
df = pd.DataFrame({'p_low':p_min_K99,
                   'p_high':p_max_K99,
                   'p_avg':p_avg_K99,
                   'F_mu+_94':F_mu_pos_K99_94,
                   'dF_mu+_94':F_pos_err_K99_94,
                   'F_mu-_94':F_mu_neg_K99_94,
                   'dF_mu-_94':F_neg_err_K99_94,
                   'F_mu+_97':F_mu_pos_K99_97,
                   'dF_mu+_97':F_mu_pos_K99_97,
                   'F_mu-_97':F_mu_neg_K99_97,
                   'dF_mu-_97':F_neg_err_K99_97,
})
df.to_csv('Muon Spectrum Tables/Kremer1999_Table1.csv', index=False)

In [7]:
def allkofer_split(a):
    b = a.replace('(','').replace(')','').replace(' ±','').replace(' x','').split(' ')
    return [float(b[0])*10**(int(b[3])), float(b[1])*10**(int(b[3])), float(b[4]), float(b[5]), float(b[6])]

In [8]:
# Measured at Kiel
Allkofer1971 = pd.read_csv('Raw Muon Tables/Allkofer1971_Table1_raw.csv')
p_min_A71, p_max_A71, p_avg_A71 = np.array([a.replace('-','').split(' ') for a in Allkofer1971['Momentum Range | Mean (GeV/c)']], dtype=float).T
F_mu_tot_A71, F_mu_err_A71, rel_err_A71, N_obs_A71, rel_accept_A71 = np.array([allkofer_split(a) for a in Allkofer1971['Differential Intensity (cm^2 sr s GeV/c)^-1 | Relative Error (%) | Number of measured particles | Relative Acceptance']]).T

In [30]:
df = pd.DataFrame({'p_low':p_min_A71,
                   'p_high':p_max_A71,
                   'p_avg':p_avg_A71,
                   'F_mu':F_mu_tot_A71,
                   'dF_mu':F_mu_err_A71,
                   'rel_err':rel_err_A71,
                   'N_obs':N_obs_A71,
                   'rel_accept':rel_accept_A71,
})
df.to_csv('Muon Spectrum Tables/Allkofer1971_Table1.csv', index=False)

In [9]:
Ayre75 = pd.read_csv('Raw Muon Tables/Ayre1975_Table8_raw.csv')
p_avg_A75 = Ayre75['Momentum (GeV/c)']
F_mu_tot_A75 = Ayre75['Differential Spectrum (s cm^2 sr (GeV/c))^-1']
F_mu_err_A75 = Ayre75['Differential Spectrum Error']
I_mu_tot_A75 = Ayre75['Integral Spectrum (s cm^2 sr)^-1']
I_mu_err_A75 = Ayre75['Integral Spectrum Error']

In [32]:
df = pd.DataFrame({'p_avg':p_avg_A75,
                   'F_mu':F_mu_tot_A75,
                   'dF_mu':F_mu_err_A75,
                   'I_mu':I_mu_tot_A75,
                   'dI_mu':I_mu_err_A75,
})
df.to_csv('Muon Spectrum Tables/Ayre1975_Table8.csv', index=False)

In [10]:
Barber68 = pd.read_csv('Raw Muon Tables/Barber1968_Table2_raw.csv')
B = np.array([b.replace('x ','').split() for b in Barber68['Momentum (GeV/c) | Differential intensity (cm^2 s sr (GeV/c))^-1 | Statistical error (%)']])
p_avg_B68 = B[:,0].astype(float)
F_mu_tot_B68 = B[:,1].astype(float)*10.**B[:,3].astype(int)
rel_err_B68 = B[:,4].astype(float)/100
F_mu_err_B68 = F_mu_tot_B68*rel_err_B68

In [33]:
df = pd.DataFrame({'p_avg':p_avg_B68,
                   'F_mu':F_mu_tot_B68,
                   'dF_mu':F_mu_err_B68,
})
df.to_csv('Muon Spectrum Tables/Barber1968_Table2.csv', index=False)

In [11]:
# Kiel-Desy at Hamburg
# Zenith angle 75 degrees (68-82 range)
Jokisch79 = pd.read_csv('Raw Muon Tables/Jokisch1979_Table2_raw.csv')
p_avg_J79 = np.array(Jokisch79['Momentum (GeV/c)'])
F_mu_75_J79 = np.array(Jokisch79['Intensity (cm^2 s sr GeV/c)^-1'])
rel_err_J79 = np.array(Jokisch79['Error (%)'])/100
N_obs_J79 = np.array(Jokisch79['Measured Number'])
F_err_75_J79 = F_mu_75_J79*rel_err_J79

In [37]:
df = pd.DataFrame({'p_avg':p_avg_J79,
                   'F_mu_75':F_mu_75_J79,
                   'dF_mu_75':F_err_75_J79,
                   'N_obs':N_obs_J79,
})
df.to_csv('Muon Spectrum Tables/Jokisch1979_Table2.csv', index=False)

In [12]:
Kellogg78 = pd.read_csv('Raw Muon Tables/Kellogg1978_Table9_raw.csv')
p_avg_K78 = np.array(Kellogg78['Momentum (GeV)'])
F_mu_75_K78 = np.array(Kellogg78['75 deg Flux (m^2 s sr GeV)^-1'])
F_err_75_K78 = np.array(Kellogg78['75 deg Error'])
F_mu_30_K78 = np.array(Kellogg78['30 deg Flux (m^2 s sr GeV)^-1'])
F_err_30_K78 = np.array(Kellogg78['30 deg Error'])

In [38]:
df = pd.DataFrame({'p_avg':p_avg_K78,
                   'F_mu_30':F_mu_30_K78,
                   'dF_mu_30':F_err_30_K78,
                   'F_mu_75':F_mu_75_K78,
                   'dF_mu_75':F_err_75_K78,
})
df.to_csv('Muon Spectrum Tables/Kellogg1978_Table9.csv', index=False)

In [13]:
Nandi72 = pd.read_csv('Raw Muon Tables/Nandi1972_Table2_raw.csv')
p_avg_N72 = np.array(Nandi72['Median Momentum (GeV/c)'])
N_obs_N72 = np.array(Nandi72['Number of Particles Observed'])
F_mu_tot_N72 = np.array(Nandi72['Differential Intensity (cm^2 s sr GeV/c)^-1'])
rel_err_N72 = (np.array(Nandi72['Statistical Error (+%)']) + np.array(Nandi72['Statistical Error (-%)']))/200
F_mu_err_N72 = F_mu_tot_N72 * rel_err_N72

In [39]:
df = pd.DataFrame({'p_avg':p_avg_N72,
                   'N_obs':N_obs_N72,
                   'F_mu':F_mu_tot_N72,
                   'dF_mu':F_mu_err_N72,
})
df.to_csv('Muon Spectrum Tables/Nandi1972_Table2.csv', index=False)

In [17]:
h_data = dict()
Phi_v_data = dict()
dPhi_v_data = dict()
n_data = dict()
Phi_data = dict()
dPhi_data = dict()
data_name = dict()
c = dict()

#Barbouti '83 data
i = 'B83'
h_data[i] = np.array([11.62, 12.12, 14.32, 19.28, 24.24, 29.06, 47.72, 52.42])-10.33 #hg/cm^2
Phi_v_data[i] = np.array([9.13, 8.87, 7.66, 5.56,4.15,3.20,1.40,1.19])*1e-3 # cm^-2 s^-1 sr^-1
dPhi_v_data[i] = np.array([0.12, 0.12, 0.09, 0.06, 0.05, 0.04, 0.02, 0.06])*1e-3
n_data[i] = F.cos_pow(h_data[i], 0)[0]
Phi_data[i] = 2*np.pi/(n_data[i]+1) * Phi_v_data[i]
dPhi_data[i] = 2*np.pi/(n_data[i]+1) * dPhi_v_data[i]
data_name[i] = "Barbouti '83 (Pb/Fe)"
c[i] = 'r'

# Bergamasco '70 data
i = 'B71'
h_data[i] = np.array([
    #50,
    106,
    158,
    238,
    #4270
])-10.33 # m.w.e.
Phi_v_data[i] = np.array([
    #2.36e-4,
    2.36e-4,
    1.20e-4,
    5.00e-5,
    # 4.79e-9
]) # cm^-2 s^-1 sr^-1
dPhi_v_data[i] = np.array([
    #0.02e-4,
    0.02e-4,
    0.01e-4,
    0.05e-5,
    # 0.23e-9
]) # cm^-2 s^-1 sr^-1
n_data[i] = F.cos_pow(h_data[i], 0)[0]
Phi_data[i] = 2*np.pi/(n_data[i]+1) * Phi_v_data[i]
dPhi_data[i] = 2*np.pi/(n_data[i]+1) * dPhi_v_data[i]
data_name[i] = "Bergamasco '71 (Rock)"
c[i] = 'g'

# Crookes '73 data (taken from various sources)
i = 'C73'
h_data[i] = np.array([
    860, # Randall and Hazen
    1510, # Bollinger
    1850,
    1584, # Barret et al.
    110, # Sreekantan and Naranan
    391,
    485,
    694,
    895,
    310, # Avan and Avan
    590,
    1290,
    833, # Miyake et al.
    1888,
    149, #Castagnoli et al.
    299,
    1319,
    826, # Achar et al.
    1881,
    764, #Krishnaswamy
    1510,
    71.6, #Stockel
    734,
    1068,
    890 # Sheldon et al.
])-10.33 # hg/cm^2
Phi_v_data[i] = np.array([
    2.17e-6, # Randall and Hazen
    3.90e-7, # Bollinger
    1.91e-7,
    3.25e-7, # Barret et al.
    2.87e-4, # Sreekantan and Naranan
    1.76e-5,
    1.24e-5,
    5.55e-5, #(sic) should probably be e-6
    2.37e-6,
    3.80e-5, # Avan and Avan
    7.3e-6,
    8.0e-7,
    2.51e-6, # Miyake et al.
    1.77e-7,
    1.70e-4, #Castagnoli et al.
    4.60e-5,
    5.60e-7,
    2.39e-6, # Achar et al.
    2.00e-7,
    2.7e-6, #Krishnaswamy
    3.4e-7,
    5.00e-4, #Stockel
    3.12e-6,
    1.03e-6,
    1.65e-6 # Sheldon et al.
]) # cm^-2 s^-1 sr^-1
dPhi_v_data[i] = np.array([
    0.02e-6, # Randall and Hazen
    0.08e-7, # Bollinger
    0.04e-7,
    0.05e-7, # Barret et al.
    0.23e-4, # Sreekantan and Naranan
    0.24e-5,
    0.13e-5,
    0.70e-5,#(sic) should probably be e-6
    0.27e-6,
    0.17e-5, # Avan and Avan
    0.4e-6,
    0.6e-7,
    0.15e-6, # Miyake et al.
    0.07e-7,
    0.04e-4, #Castagnoli et al.
    0.01e-5,
    0.07e-7,
    0.15e-6, # Achar et al.
    0.12e-7,
    0.1e-6, #Krishnaswamy
    0.1e-7,
    0.17e-4, #Stockel
    0.15e-6,
    0.04e-6,
    0.09e-6 # Sheldon et al.
]) # cm^-2 s^-1 sr^-1
n_data[i] = F.cos_pow(h_data[i], 0)[0]
Phi_data[i] = 2*np.pi/(n_data[i]+1) * Phi_v_data[i]
dPhi_data[i] = 2*np.pi/(n_data[i]+1) * dPhi_v_data[i]
data_name[i] = "Crookes '73 (Compilation)"
c[i] = 'orange'

#Rogers '84 data
i = 'R84'
h_data[i] = np.array([10.3, 18.0, 20.9, 38.4, 45.6, 59.0, 71.7, 100, 111, 143, 175]) #hg/cm^2
Phi_v_data[i] = np.array([3.70e-3, 2.24e-3, 1.83e-3, 9.61e-4, 7.59e-4, 4.99e-4, 3.90e-4, 2.02e-4, 1.80e-4, 1.10e-4, 7.39e-5]) #sr^-1 cm^-2 s^-1
dPhi_v_data[i] = np.array([0.32e-3, 0.15e-3, 0.18e-3, 0.85e-4, 0.72e-4, 0.41e-4, 0.33e-4, 0.18e-4, 0.16e-4, 0.10e-4, 0.88e-5])
n_data[i] = F.cos_pow(h_data[i], 0)[0]
Phi_data[i] = 2*np.pi/(n_data[i]+1) * Phi_v_data[i]
dPhi_data[i] = 2*np.pi/(n_data[i]+1) * dPhi_v_data[i]
data_name[i] = "Rogers '84 (Water)"
c[i] = 'b'

In [42]:
f_names = {'B83':'Barbouti1983_Table8',
           'B71':'Bergamasco1971_Table1',
           'C73':'Crookes1973_Table1', 
           'R84':'Rogers1984_Table3'
          }

for i in ['B83', 'B71', 'C73', 'R84']:
    df = pd.DataFrame({'h':h_data[i],
                       'Phi_v':Phi_v_data[i],
                       'dPhi_v':dPhi_v_data[i],
    })
    df.to_csv('Muon Spectrum Tables/{}.csv'.format(f_names[i]), index=False)